## 3-Stage Pipeline: Boxed-Only SFT → CoT SFT → GRPO

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260324+cu128 requires torch==2.12.0.dev20260324, but you have torch 2.10.0 which is incompatible.
cuda-python 12.9.6 requires cuda-bindings~=12.9.6, but you have cuda-bindings 12.9.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
goo

In [2]:
import datasets
import trl
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)

datasets: 4.3.0
trl: 0.24.0


In [3]:
import os
import sys
import stat
import shutil
import gc
import zipfile
import types
import re
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Block Mamba3 import chain ---
_mock_class = type("_Mock", (), {})
for _name, _attrs in [
    ("mamba_ssm.modules.mamba3", {"Mamba3": _mock_class}),
    ("mamba_ssm.ops.cute", {}),
    ("mamba_ssm.ops.cute.mamba3", {}),
    ("mamba_ssm.ops.cute.mamba3.mamba3_step_fn", {"mamba3_step_fn": lambda *a, **kw: None}),
]:
    if _name not in sys.modules:
        _m = types.ModuleType(_name)
        _m.__path__ = []
        for _k, _v in _attrs.items():
            setattr(_m, _k, _v)
        sys.modules[_name] = _m

# --- Triton rmsnorm fallback ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst

# ===================== HYPERPARAMETERS =====================
# Stage 1: Boxed-only SFT (answer-only, loss only on \boxed{})
S1_SAMPLES = 600         # answer-only samples for boxed-only loss
S1_EPOCHS = 1
S1_LR = 2e-4

# Stage 2: Ultra-compact rule SFT (7-32 char rules, full-text loss)
S2_SAMPLES = 600         # compact rule samples (from 7945 pool)
S2_EPOCHS = 1
S2_LR = 5e-5            # lower LR to preserve Stage 1 gains

# Stage 3: DISABLED (no GRPO, pure SFT test)
# S3_SAMPLES = 0

# Shared
LORA_RANK = 32
MAX_SEQ_LEN = 1024
GRAD_ACCUM = 4

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✓ Hyperparameters set")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


✓ Hyperparameters set


## Stage 1: Boxed-Only Loss SFT
Answer-only training with loss ONLY on `\boxed{answer}` tokens.
This teaches format without disturbing the model's thinking ability.

In [4]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

# </think> special token ID — used to find where boxed answer starts
THINK_CLOSE_ID = tokenizer.convert_tokens_to_ids("</think>")
print(f"</think> token ID: {THINK_CLOSE_ID}")

# Load answer-only data (random 600 from train.csv, same as V2/E1)
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
train_df = train_df.sample(n=min(S1_SAMPLES, len(train_df)), seed=42)
print(f"Stage 1 data: {len(train_df)} samples")

hf_s1 = Dataset.from_pandas(train_df.to_pandas())

def build_boxed_only_example(example):
    # Pre-tokenize with labels: mask everything up to and including </think>.
    prompt = example["prompt"]
    answer = example["answer"]
    user_msg = prompt + SUFFIX
    assistant_msg = f"\\boxed{{{answer}}}"
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
        enable_thinking=True,
    )
    input_ids = tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LEN)
    
    # Find </think> token and mask everything up to and including it
    prefix_len = len(input_ids)  # default: mask all (safety)
    for i, tid in enumerate(input_ids):
        if tid == THINK_CLOSE_ID:
            prefix_len = i + 1
            break
    labels = [-100] * prefix_len + input_ids[prefix_len:]
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

hf_s1 = hf_s1.map(build_boxed_only_example, remove_columns=hf_s1.column_names)
n_loss = sum(1 for x in hf_s1[0]["labels"] if x != -100)
print(f"Stage 1 ready: {len(hf_s1)} examples, {n_loss} loss tokens in example 0")
print(f"Loss tokens: {tokenizer.decode([t for t in hf_s1[0]['labels'] if t != -100])}")

</think> token ID: 13
Stage 1 data: 600 samples


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Stage 1 ready: 600 examples, 12 loss tokens in example 0
Loss tokens: \boxed{11.11}<|im_end|>



In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=16, target_modules="all-linear",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded. Vocab size: 131072
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


In [6]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

tokenizer.model_max_length = MAX_SEQ_LEN

s1_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=S1_EPOCHS,
    learning_rate=S1_LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    dataset_kwargs={"skip_prepare_dataset": True},
)

s1_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_s1,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    args=s1_args,
)

print("=== Stage 1: Boxed-Only SFT ===")
s1_trainer.train()
print("✓ Stage 1 complete")

del s1_trainer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after Stage 1: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


=== Stage 1: Boxed-Only SFT ===


Step,Training Loss
5,2.684050
10,2.749229
15,2.162399
20,2.334897
25,1.875302
30,2.393744
35,1.706746
40,2.461305
45,2.083467
50,2.236247


✓ Stage 1 complete
GPU after Stage 1: 62.2 GB


## Stage 2: CoT SFT
Rule-generated programmatic CoT. Full-text loss (standard SFT).
Lower LR to preserve Stage 1 format training.

In [7]:
# Load ultra-compact rule data (7-32 char rules per type, 7945 total)
cot_df = pl.read_csv('/kaggle/input/prog-cot-training-data/sft_compact_rules.csv')
cot_df = cot_df.sample(n=min(S2_SAMPLES, len(cot_df)), seed=42)
print(f"Stage 2 data: {len(cot_df)} samples")

hf_s2 = Dataset.from_pandas(cot_df.to_pandas())

SHORT_SUFFIX = "\nPut your final answer inside \\boxed{}."

def build_cot_text(example):
    # Use reasoning_content field to avoid double <think> blocks.
    # enable_thinking=True auto-inserts <think>...</think> around reasoning_content.
    prompt = example["prompt"]
    answer = example["answer"]
    thinking = example.get("thinking", "") or ""
    user_msg = prompt + SHORT_SUFFIX
    
    try:
        if thinking.strip():
            # Use reasoning_content so template puts it inside <think> properly
            messages = [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": f"\\boxed{{{answer}}}", "reasoning_content": thinking},
            ]
        else:
            messages = [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": f"\\boxed{{{answer}}}"},
            ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=True
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{thinking}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
        )
    return {"text": text}

hf_s2 = hf_s2.map(build_cot_text, remove_columns=hf_s2.column_names)
print(f"Stage 2 ready: {len(hf_s2)} examples")
print(f"Example (first 300 chars):\n{hf_s2[0]['text'][:300]}")

s2_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=S2_EPOCHS,
    learning_rate=S2_LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

s2_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_s2,
    processing_class=tokenizer,
    args=s2_args,
)

print("=== Stage 2: CoT SFT ===")
s2_trainer.train()
print("✓ Stage 2 complete")

del s2_trainer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after Stage 2: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

Stage 2 data: 600 samples


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Stage 2 ready: 600 examples
Example (first 300 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
18 -> XVIII
38 -> XXXVIII
83 -> LXXXIII
Now, write the number 87 in the Wonderland numeral system.
Put your final answer inside \boxed{


Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

=== Stage 2: CoT SFT ===


Step,Training Loss
5,10.893282
10,11.143230
15,10.189874
20,7.781016
25,6.916814
30,7.304234
35,4.876735
40,7.462595
45,4.568038
50,4.696367


✓ Stage 2 complete
GPU after Stage 2: 62.2 GB


## Save and Package Submission

In [8]:
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip ready!")

Adapter files saved to /kaggle/working/adapter:
  README.md (5.2 KB)
  adapter_model.safetensors (3454393.7 KB)
  adapter_config.json (1.1 KB)

Created /kaggle/working/submission.zip (3046.2 MB)
Contents: ['README.md', 'adapter_model.safetensors', 'adapter_config.json']
✓ submission.zip ready!
